# GKBA Results Explorer
Load output from any simulation script and plot currents, spin density, charge density.

In [ ]:
using DelimitedFiles, LinearAlgebra
using PyPlot
const plt = PyPlot
plt.rc("text", usetex=true)
plt.rc("axes", linewidth=1);


## Parameters — edit here

In [ ]:
# Change these to match the simulation you want to explore
name       = "gkba_krep_prece"   # matches name in the run_*.jl script
data_path  = "../data"
nx, ny     = 2, 1
nσ         = 2
t_step     = 0.1
hbar       = 1.0;  # natural units


## Load data

In [ ]:
p(f) = joinpath(data_path, "__jl.txt")

cc_raw     = readdlm(p("cc"))
sneq_raw   = readdlm(p("sneq"))
sc_raw     = readdlm(p("sc"))
cspins_raw = readdlm(p("cspins"))
cden_raw   = readdlm(p("cden"))
rho_raw    = readdlm(p("rho"), ",", ComplexF64)

ns       = nx * ny
n_spin   = ns * nσ
nt       = size(cc_raw, 1) ÷ 2
ts       = t_step:t_step:nt*t_step

# Reshape into (time, site, component) arrays
cc_αt    = zeros(Float64, 2, nt)
cc_αt[1,:] = cc_raw[1:2:end]
cc_αt[2,:] = cc_raw[2:2:end]

sneq_ta1x = zeros(Float64, nt, ns, 3)
for i in 1:ns
    sneq_ta1x[:,i,:] = sneq_raw[i:ns:end, :]
end

sc_αxt = zeros(Float64, 2, 3, nt)
for x in 1:3
    sc_αxt[1,x,:] = sc_raw[x:3:end, 1]
    sc_αxt[2,x,:] = sc_raw[x:3:end, 2]
end

cden_ta1 = zeros(Float64, nt, n_spin)
for i in 1:n_spin
    cden_ta1[:,i] = cden_raw[i:n_spin:end]
end

cspins_tax = zeros(Float64, nt, ns, 3)
for i in 1:ns
    cspins_tax[:,i,:] = cspins_raw[i:ns:end, :]
end

println("Loaded  time steps");


## Charge current

In [ ]:
fs = 18
fig, ax = plt.subplots(1, 1)
ax.plot(ts, cc_αt[1,:]*4π, label=raw"$",  color="steelblue", lw=1)
ax.plot(ts, cc_αt[2,:]*4π, label=raw"$",  color="tomato",    lw=1)
ax.set_xlabel(raw"$",                 fontsize=fs)
ax.set_ylabel(raw"\,(e\gamma/h)$",   fontsize=fs)
ax.tick_params(labelsize=fs-2, direction="in")
ax.ticklabel_format(axis="y", style="sci", scilimits=(-1,2), useMathText=true)
plt.legend(frameon=false, fontsize=fs-4)
plt.tight_layout()
plt.show()


## Spin density (site 1)

In [ ]:
fs   = 18
site = 1
fig, ax = plt.subplots(1, 1)
ax.plot(ts, sneq_ta1x[:,site,1], label=raw"^x$")
ax.plot(ts, sneq_ta1x[:,site,2], label=raw"^y$", color="green", ls="--")
ax.plot(ts, sneq_ta1x[:,site,3], label=raw"^z$", color="red",   ls=":")
ax.set_xlabel(raw"$",                              fontsize=fs)
ax.set_ylabel(raw"$\langle S^lpha_iangle$",   fontsize=fs)
ax.tick_params(labelsize=fs-2, direction="in")
ax.ticklabel_format(axis="y", style="sci", scilimits=(-1,2), useMathText=true)
plt.legend(frameon=false, fontsize=fs-4)
plt.tight_layout()
plt.show()


## Spin current (lead L)

In [ ]:
fs = 18
fig, ax = plt.subplots(1, 1)
ax.plot(ts, sc_αxt[1,1,:]/π, label=raw"^{S_x}_L$")
ax.plot(ts, sc_αxt[1,2,:]/π, label=raw"^{S_y}_L$", color="green", ls=":")
ax.plot(ts, sc_αxt[1,3,:]/π, label=raw"^{S_z}_L$", color="red",   ls=":")
ax.set_xlabel(raw"$",                              fontsize=fs)
ax.set_ylabel(raw"^{S_lpha}_L\,(e\gamma/h)$",  fontsize=fs)
ax.tick_params(labelsize=fs-2, direction="in")
ax.ticklabel_format(axis="y", style="sci", scilimits=(-1,2), useMathText=true)
plt.legend(frameon=false, fontsize=fs-4)
plt.tight_layout()
plt.show()


## Charge density

In [ ]:
fs = 18
fig, ax = plt.subplots(1, 1)
for i in 1:ns
    # sum over spin channels for each site
    n_i = cden_ta1[:, 2i-1] + cden_ta1[:, 2i]
    ax.plot(ts, n_i, label="site ")
end
ax.set_xlabel(raw"$",                          fontsize=fs)
ax.set_ylabel(raw"$\langle n_i angle$",       fontsize=fs)
ax.tick_params(labelsize=fs-2, direction="in")
ax.ticklabel_format(axis="y", style="sci", scilimits=(-1,2), useMathText=true)
plt.legend(frameon=false, fontsize=fs-4)
plt.tight_layout()
plt.show()


## Classical spin trajectory

In [ ]:
fs   = 18
site = 1
fig, ax = plt.subplots(1, 1)
ax.plot(ts, cspins_tax[:,site,1], label=raw"^x$")
ax.plot(ts, cspins_tax[:,site,2], label=raw"^y$", ls="--")
ax.plot(ts, cspins_tax[:,site,3], label=raw"^z$", ls=":")
ax.set_xlabel(raw"$",        fontsize=fs)
ax.set_ylabel(raw"^lpha$", fontsize=fs)
ax.tick_params(labelsize=fs-2, direction="in")
plt.legend(frameon=false, fontsize=fs-4)
plt.tight_layout()
plt.show()


## Density matrix ρ(t) = -i G^<_s(t)

In [ ]:
# Reshape rho_raw into time series of density matrices
rho_t = [reshape(rho_raw[i,:], (n_spin, n_spin)) for i in 1:nt]

# Example: trace of density matrix (total electron number, should be conserved)
n_tot = [-1im * tr(rho_t[i]) for i in 1:nt]
fig, ax = plt.subplots(1, 1)
ax.plot(ts, real.(n_tot))
ax.set_xlabel(raw"$",                  fontsize=18)
ax.set_ylabel(raw"{m tot}$",        fontsize=18)
ax.tick_params(labelsize=16, direction="in")
plt.tight_layout()
plt.show()
